# NB03.5 — Síntesis Metodológica y Documentación
### VíaSegura AI — Priorización espacial de siniestralidad vial en Bogotá

---

## Sección 0 — Propósito y cómo usar este notebook

**Este notebook es de referencia, no computacional.** No genera outputs nuevos (sin CSVs, sin mapas nuevos). Su propósito es:

1. Documentar con precisión qué se hizo en NB01, NB02 y NB03.
2. Explicar cada variable derivada y la fórmula exacta con la que se calculó.
3. Comparar los distintos índices de criticidad usados en el proyecto y sus límites metodológicos.
4. Servir como punto de partida para cualquier persona (o sesión futura de trabajo) que retome el proyecto.

**Cómo navegarlo:** Ejecuta con *Run All* para cargar las tablas reales. Las celdas de código solo leen archivos existentes en `outputs/reports/` — ninguna descarga datos ni escribe archivos nuevos.

> **Nota sobre consistencia:** El NB02 tuvo un bug (ADR-06) corregido el 2026-05-08 via `fix_ipi_nb02.py`. El NB03 fue ejecutado antes de esa corrección. El delta máximo en valores IPI es ~0.03 puntos, lo que no afecta la composición del Top 200 reciente usado en NB04. La recomendación es reejecutar NB03 antes de NB05 para consistencia total.

In [1]:
# ── SETUP DE RUTAS ── Importa desde config.py (compatible VS Code + JupyterLab)
import sys
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

_root = next(
    (c for c in [Path.cwd(), Path.cwd().parent] if (c / 'config.py').exists()),
    None
)
if _root is None:
    raise RuntimeError(f"No se encontró config.py. cwd={Path.cwd()}")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROJECT_ROOT, DATA_RAW, DATA_PROCESSED, REPORTS, MAPS

PROCESSED = DATA_PROCESSED  # alias usado en este notebook

print(f"✓ PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"✓ REPORTS:        {REPORTS}")
print(f"✓ DATA_PROCESSED: {DATA_PROCESSED}")

✓ PROJECT_ROOT:   C:\Users\jorge\Documents\viasegura_ai
✓ REPORTS:        C:\Users\jorge\Documents\viasegura_ai\outputs\reports
✓ DATA_PROCESSED: C:\Users\jorge\Documents\viasegura_ai\data\processed


---

## Sección 1 — Contexto del proyecto VíaSegura AI

### Frase central

> **VíaSegura AI usa datos oficiales de siniestralidad vial de Bogotá para identificar patrones espaciales y temporales, clasificar zonas críticas y generar recomendaciones preliminares de intervención para apoyar decisiones de seguridad vial urbana.**

### Qué es y qué NO es

| ✅ SÍ es | ❌ NO es |
|---|---|
| Herramienta exploratoria de priorización | Modelo predictivo de accidentes |
| Identificador de zonas con alta concentración histórica de siniestros | Evaluación definitiva de riesgo vial |
| Proyecto de portafolio profesional defendible técnicamente | Producto comercial terminado |
| Base para un producto futuro aplicado al sector transporte en Colombia | Diagnóstico actualizado del Bogotá de 2026 |

### Por qué Bogotá y por qué 2016–2019

**Bogotá:** es la ciudad con más siniestros reportados del país, tiene una infraestructura de datos abiertos relativamente madura (SDM/SIMUR/ArcGIS) y una geografía manejable para un MVP antes de pensar en escala nacional.

**2016–2019:** periodo cerrado, completo, pre-pandemia y con ~65,000 registros/año estables. Se evitaron:
- 2025–2026: incompletos (año en curso).
- 2022–2025: ratio de gravedad 11–14× por encima del periodo base → cambio de metodología de registro en SIMUR (ver ADR-10).
- 2020–2021: usados como **periodo de validación**, no como base, por el efecto COVID-19 en la movilidad.

### Audiencia objetivo

1. **Académica / portafolio:** profesores, evaluadores, recruiters técnicos.
2. **Profesional:** secretarías de movilidad, consultoras de transporte, ANSV.
3. **Comunidad open source:** GitHub público.

---

## Sección 2 — Fuente de datos: SIMUR / ArcGIS FeatureServer

### Acceso a la API

Todos los datos del proyecto provienen de la API ArcGIS de la Secretaría Distrital de Movilidad (SDM) de Bogotá, publicada a través del SIMUR:

```
URL base: https://sig.simur.gov.co/arcgis/rest/services/Accidentalidad/AccidentalidadAnalisis/FeatureServer
```

Se eligió esta fuente sobre `datos.gov.co` porque la SDM actualiza el FeatureServer directamente y permite consultas filtradas por campo (ADR-01).

### Capas disponibles en el FeatureServer

| Layer ID | Nombre | Registros totales | Clave de join | Uso en el proyecto |
|---|---|---|---|---|
| 0 | MUERTO | ~15,600 | FORMULARIO | Integrado indirectamente via GRAVEDAD='CON MUERTOS' |
| 1 | LESIONADO | ~475,000 | FORMULARIO | Integrado indirectamente via GRAVEDAD='CON HERIDOS' |
| **2** | **ACCIDENTE** | **~904,424** | FORMULARIO | **Capa principal — usada en NB01, NB02 y NB03** |
| 3 | VM_ACC_ACTOR_VIAL | ~3,267,086 | FORMULARIO | Planificado para NB04 — tipo de actor (peatón, moto, ciclista...) |
| 4 | VM_ACC_CAUSA | ~1,865,722 | FORMULARIO | Planificado para NB04 — causa más frecuente |
| 5 | VM_ACC_VEHICULO | ~2,803,601 | FORMULARIO | Planificado para NB04 — tipo de vehículo |
| 6 | VM_ACC_VIA | ~961,101 | FORMULARIO | Pendiente verificación (devolvió 0 en muestra de NB03) |

### Cómo se hace el conteo por año (patrón base)

```python
import requests

url = 'https://sig.simur.gov.co/arcgis/rest/services/Accidentalidad/AccidentalidadAnalisis/FeatureServer/2/query'

# Conteo total
r = requests.get(url, params={'where': '1=1', 'returnCountOnly': 'true', 'f': 'json'})
total = r.json()['count']  # → 904,424

# Conteo por año
r = requests.get(url, params={'where': 'ANO_OCURRENCIA_ACC = 2019', 'returnCountOnly': 'true', 'f': 'json'})
count_2019 = r.json()['count']  # → ~64,000
```

> **Nota de reproducibilidad:** el FeatureServer es una fuente viva. SIMUR puede actualizar datos históricos sin aviso. Para blindar la reproducibilidad se creó `scripts/generar_snapshot_metadata.py`, que guarda el hash MD5 de cada archivo raw en `data/raw/_snapshot_metadata.json`.

---

## Sección 3 — NB01: Exploración y limpieza de datos

**Objetivo del NB01:** conectar a SIMUR, auditar la disponibilidad de datos por año, descargar el periodo base 2016–2019 de forma segura y construir una base limpia y validada.

**Resultado:** `data/processed/accidentes_bogota_2016_2019_limpio.csv` — 260,831 registros × 19 columnas.

---

### 3.1 — Variables del dataset original de SIMUR (capa ACCIDENTE)

La capa devuelve 22+ campos. Se seleccionaron 18 para el análisis:

| Variable | Descripción | Tipo | Transformación aplicada |
|---|---|---|---|
| `OBJECTID` | ID único del registro en ArcGIS | Int | Ninguna (clave primaria) |
| `FORMULARIO` | Código del formulario policial | String | Ninguna (clave de join con capas hermanas) |
| `CODIGO_ACCIDENTE` | Código del accidente | Int | Ninguna |
| `FECHA_OCURRENCIA_ACC` | Timestamp en milisegundos epoch | Int → datetime | `pd.to_datetime(unit='ms', errors='coerce')` |
| `HORA_OCURRENCIA_ACC` | Hora en formato HH:MM:SS | String | Ninguna |
| `ANO_OCURRENCIA_ACC` | Año del siniestro | Int | Ninguna |
| `MES_OCURRENCIA_ACC` | Mes del siniestro | String | `.str.strip().str.upper()` |
| `DIA_OCURRENCIA_ACC` | Día de la semana | String | `.str.strip().str.upper()` |
| `DIRECCION` | Dirección textual del siniestro | String | `.str.strip().str.upper()` |
| `GRAVEDAD` | Severidad: SOLO DAÑOS / CON HERIDOS / CON MUERTOS | String | `.str.strip().str.upper()` |
| `CLASE_ACC` | Tipo: CHOQUE / ATROPELLO / VOLCAMIENTO / etc. | String | `.str.strip().str.upper()` |
| `LOCALIDAD` | Localidad administrativa de Bogotá | String | `.str.strip().str.upper()` |
| `MUNICIPIO` | Municipio (todos = 'BOGOTA DC') | String | `.str.strip().str.upper()` |
| `LATITUD` | Coordenada Y (esperado: 4.x) | Float | `pd.to_numeric(errors='coerce')` + filtro bbox |
| `LONGITUD` | Coordenada X (esperado: -74.x) | Float | `pd.to_numeric(errors='coerce')` + filtro bbox |
| `BARRIO` | Barrio del siniestro | String | `.str.strip().str.upper()` |
| `MVINOMBRE` | Nombre de la vía principal | String | `.str.strip().str.upper()` |
| `DISTANCIA_VIA` | Distancia al eje de vía (metros) | Float | Ninguna |

**Variable derivada:** `puntaje_gravedad` (ver Sección 3.4).

---

### 3.2 — Descarga segura por bloques

Un primer intento de descarga con bloques de 2,000 registros falló a las ~196,000 filas con `RemoteDisconnected` (D4). La solución implementada:

```python
def descargar_accidentes_por_anio_seguro(url_base, anio, registros_por_bloque=1000, max_reintentos=5):
    """
    Descarga un año completo en bloques de 1,000 registros.
    Guarda cada bloque como CSV individual para permitir reanudación.
    Si un bloque ya existe, lo omite (idempotente).
    """
    ...
```

**Resultado:** 262 chunks en `data/raw/chunks_accidentes_2016_2019/`, luego unidos con `pd.concat`. La unión produjo exactamente 260,831 filas — coincide con el conteo oficial del servidor (diferencia = 0).

---

### 3.3 — Transformaciones de limpieza

Se aplicaron sobre una copia (`df_limpia = df_raw.copy()`) para no alterar el raw:

1. **Conversión de fecha:** epoch ms → datetime de pandas.
2. **Normalización de texto:** `.str.strip().str.upper()` en 9 columnas categóricas.
3. **Filtro bounding box Bogotá:**
   ```python
   df_limpia = df_limpia.dropna(subset=['LATITUD', 'LONGITUD'])
   df_limpia = df_limpia[
       (df_limpia['LATITUD'] > 4.0) & (df_limpia['LATITUD'] < 5.0) &
       (df_limpia['LONGITUD'] > -75.0) & (df_limpia['LONGITUD'] < -73.0)
   ]
   ```
   **Resultado: 0 filas eliminadas** — la fuente ya venía limpia.

---

### 3.4 — `puntaje_gravedad`: primera variable derivada

```python
puntaje_gravedad = {
    'SOLO DANOS': 1,   # variante sin tilde también aceptada
    'SOLO DAÑOS': 1,
    'CON HERIDOS': 3,
    'CON MUERTOS': 5,
}
df_limpia['puntaje_gravedad'] = df_limpia['GRAVEDAD'].map(puntaje_gravedad)
# Validación: 0 nulos en puntaje_gravedad
```

**Qué significa esta escala:** es una escala ordinal simple que asigna un peso relativo a cada tipo de siniestro. 5 vale más que 3, que vale más que 1. **No es una escala de intervalos** — la distancia entre 1 y 3 no es necesariamente igual a la de 3 a 5.

**Límite conocido (L5):** con esta escala, un siniestro fatal equivale a solo 5 siniestros materiales. Con la escala EPDO de la NHTSA (estándar EE.UU.) equivaldría a 24. Con costos económicos (FHWA) equivaldría a 542. Esto significa que el ranking del IPI puede sub-priorizar zonas con alta fatalidad relativa. Ver comparación completa en Sección 6.

### 3.5 — Resultados del NB01 (datos reales)

In [2]:
# Resumen de calidad de la base limpia
calidad = pd.read_csv(REPORTS / 'resumen_calidad_datos_2016_2019.csv')
print('=== Resumen de calidad — base 2016–2019 ===')
display(calidad)

=== Resumen de calidad — base 2016–2019 ===


,base,periodo,filas_raw,filas_limpias,filas_eliminadas_limpieza,duplicados_objectid_raw,duplicados_objectid_limpia,nulos_latitud,nulos_longitud,nulos_puntaje_gravedad
0,accidentes_bogota_2016_2019_limpio,2016-2019,260831,260831,0,0,0,0,0,0


In [3]:
# Top 5 localidades por criticidad total
localidades = pd.read_csv(REPORTS / 'tabla_localidad_2016_2019.csv')
print('=== Top 10 localidades por criticidad total (periodo base 2016–2019) ===')
display(localidades.head(10))

=== Top 10 localidades por criticidad total (periodo base 2016–2019) ===


,LOCALIDAD,cantidad_siniestros,criticidad_total,criticidad_promedio
0,KENNEDY,30716,57196,1.8621
1,ENGATIVA,27782,45642,1.6429
2,SUBA,24672,41484,1.6814
3,USAQUEN,25806,37054,1.4359
4,FONTIBON,22135,33835,1.5286
5,PUENTE ARANDA,17872,31052,1.7375
6,CHAPINERO,16834,24594,1.4610
7,BOSA,11654,23442,2.0115
8,TEUSAQUILLO,13694,21562,1.5746
9,BARRIOS UNIDOS,13681,21385,1.5631


In [4]:
# Distribución de gravedad
gravedad = pd.read_csv(REPORTS / 'tabla_gravedad_2016_2019.csv')
print('=== Distribución por gravedad — periodo base 2016–2019 ===')
display(gravedad)

=== Distribución por gravedad — periodo base 2016–2019 ===


,GRAVEDAD,cantidad_siniestros,criticidad_total
0,CON HERIDOS,84730,254190
1,SOLO DANOS,171904,171904
2,CON MUERTOS,4197,20985


---

## Sección 4 — NB02: Índice de Prioridad de Intervención (IPI)

**Objetivo del NB02:** construir un índice multidimensional para pasar del mapa descriptivo de siniestros (NB01) a una lista rankeada y metodológicamente defendible de zonas prioritarias.

**Resultado principal:** `outputs/reports/top50_IPI_final_2016_2019.csv` y `outputs/maps/mapa_top50_IPI_final_2016_2019.html`.

---

### 4.1 — Agregación espacial: grilla de 0.001°

Los 260,831 siniestros son puntos individuales. Para identificar *zonas* críticas hay que agruparlos en unidades espaciales comparables. Se eligió redondear las coordenadas a 3 decimales (D6, D12):

```python
df['LAT_ZONA'] = df['LATITUD'].round(3)
df['LON_ZONA'] = df['LONGITUD'].round(3)
```

**¿Qué tan grande es una celda de 0.001°?**
- En Bogotá (latitud ~4.6°N), 0.001° de latitud ≈ 111 m y 0.001° de longitud ≈ 110 m.
- Cada celda es aproximadamente una manzana grande o una intersección con su entorno inmediato.

**Resultado:** 260,831 siniestros → **17,130 zonas únicas**.

**Limitación conocida (L4):** la grilla es arbitraria. No respeta la red vial ni las intersecciones reales. Una intersección vial puede caer en la frontera entre 2 o 4 celdas y repartirse entre ellas, diluyendo su señal. Alternativas no evaluadas: H3 (hexágonos Uber), snap-to-network, DBSCAN.

---

### 4.2 — Variables intermedias por zona (agregación)

Para cada celda `(LAT_ZONA, LON_ZONA)` se calculó:

| Variable | Fórmula | Notas |
|---|---|---|
| `cantidad_siniestros` | `COUNT(OBJECTID)` | Volumen total de siniestros |
| `criticidad_total` | `SUM(puntaje_gravedad)` | Acumulación ponderada por gravedad |
| `criticidad_promedio` | `MEAN(puntaje_gravedad)` | Severidad típica por siniestro |
| `anios_activos` | `NUNIQUE(ANO_OCURRENCIA_ACC)` | Años con al menos 1 siniestro. Rango: 1–4. |
| `siniestros_solo_danos` | `COUNT(GRAVEDAD='SOLO DAÑOS')` | Siniestros solo materiales |
| `siniestros_con_heridos` | `COUNT(GRAVEDAD='CON HERIDOS')` | Siniestros con heridos |
| `siniestros_con_muertos` | `COUNT(GRAVEDAD='CON MUERTOS')` | Siniestros fatales |
| `localidad_predominante` | `MODA(LOCALIDAD)` | Localidad más frecuente en la zona |
| `barrio_predominante` | `MODA(BARRIO)` | Barrio más frecuente |
| `via_predominante` | `MODA(MVINOMBRE)` | Vía más frecuente |
| `clase_predominante` | `MODA(CLASE_ACC)` | Tipo de siniestro más frecuente |
| `gravedad_predominante` | `MODA(GRAVEDAD)` | Gravedad más frecuente |

**Función auxiliar usada:**
```python
def valor_mas_frecuente(serie):
    moda = serie.mode()
    return moda.iloc[0] if len(moda) > 0 else None
```

---

### 4.3 — Los 5 scores del IPI (fórmulas exactas)

El IPI combina 5 dimensiones independientes de criticidad. Cada una se normaliza por separado antes de promediarlas:

#### Score 1: `score_volumen`
```python
score_volumen = zonas['cantidad_siniestros'].rank() / n_zonas
```
- **Qué mide:** cuántos siniestros tuvo la zona en relación al universo completo.
- **Normalización:** percentil de rango. Una zona con `score_volumen = 0.95` supera en volumen al 95% de las 17,130 zonas.
- **Rango:** 0–1.

#### Score 2: `score_criticidad_total`
```python
score_criticidad_total = zonas['criticidad_total'].rank() / n_zonas
```
- **Qué mide:** la carga histórica acumulada ponderada por gravedad. Combina volumen y gravedad en un solo número.
- **Diferencia con score_volumen:** una zona con 100 siniestros mortales (criticidad = 500) puede tener mayor score que una con 300 siniestros materiales (criticidad = 300), aunque tenga menos siniestros.
- **Rango:** 0–1.

#### Score 3: `score_severidad_promedio`
```python
score_severidad_promedio = zonas['criticidad_promedio'].rank() / n_zonas
```
- **Qué mide:** la gravedad típica de los siniestros de la zona, independientemente del volumen.
- **Caso de uso:** detecta zonas con pocos siniestros pero muy graves (atropellos fatales en vías de alta velocidad).
- **Rango:** 0–1.

#### Score 4: `score_persistencia`
```python
score_persistencia = zonas['anios_activos'] / 4  # 4 años en el periodo base
```
- **Qué mide:** fracción de años del periodo en que la zona tuvo al menos un siniestro.
- **Normalización:** ratio directo, **no** percentil de rango. El divisor es el número de años del periodo (4 para NB02, 2 para NB03).
- **Por qué ratio directo y no percentil:** `anios_activos` solo toma 4 valores discretos (1, 2, 3, 4). Un percentil sobre 4 valores produciría una escala artificial. El ratio directo preserva el significado semántico (0.25 = activa 1 de 4 años; 1.0 = activa todos los años).
- **Valores posibles:** 0.25, 0.50, 0.75, 1.00.

#### Score 5: `score_fatalidad`
```python
score_fatalidad = zonas['siniestros_con_muertos'].rank() / n_zonas
```
- **Qué mide:** la posición relativa de la zona en número de siniestros fatales.
- **Caso de uso:** garantiza que las zonas con muertos siempre reciban un score diferencial, incluso si su volumen total es bajo.
- **Rango:** 0–1.

#### ¿Por qué percentil de rango y no normalización min-max?

La normalización min-max (`(x - min) / (max - min)`) es sensible a valores extremos. Si una zona tiene 500 siniestros y el resto tiene menos de 200, la normalización min-max comprime el 99% de las zonas en el rango 0–0.4. El percentil de rango distribuye uniformemente el espacio 0–1 entre todas las zonas, sin importar los outliers (D14).

---

### 4.4 — Cálculo del IPI

```python
scores_cols = [
    'score_volumen',
    'score_criticidad_total',
    'score_severidad_promedio',
    'score_persistencia',
    'score_fatalidad'
]

zonas['IPI'] = zonas[scores_cols].mean(axis=1) * 100
zonas['rank_IPI'] = zonas['IPI'].rank(ascending=False).astype(int)
```

**Características del IPI:**
- Rango teórico: 0–100.
- Pesos: **iguales** (20% por score). No hay jerarquización a priori entre dimensiones (D13).
- El Top 50 del periodo base tiene IPI entre **93.27 y 96.25**.

**Bug ADR-06 — corregido el 2026-05-08:**
Una celda del NB02 original usaba pesos implícitos desiguales (criticidad×0.25, fatalidad×0.15) en vez de `mean()`. Se detectó y corrigió via `fix_ipi_nb02.py`. El delta máximo en valores IPI fue ~0.03 puntos. Todos los outputs del NB02 fueron regenerados con la corrección.

---

### 4.5 — Clasificaciones derivadas del IPI

#### a) `prioridad_IPI` — por posición en el ranking

| Rango de rank | Etiqueta |
|---|---|
| rank ≤ 50 | Prioridad 1 — Intervención prioritaria |
| rank ≤ 200 | Prioridad 2 — Auditoría de seguridad vial |
| rank ≤ 500 | Prioridad 3 — Monitoreo y gestión preventiva |
| rank > 500 | Seguimiento |

#### b) `familia_analitica` — por perfil de scores

```python
def clasificar_familia_analitica(row):
    if row['rank_IPI'] <= 200 and row['rank_criticidad_total'] <= 200:
        return 'Hotspot robusto integral'
    elif row['rank_IPI'] <= 200 and row['rank_muertes'] <= 200:
        return 'Hotspot de severidad/fatalidad'
    elif row['rank_criticidad_total'] <= 200 and row['rank_IPI'] > 200:
        return 'Hotspot de carga acumulada'
    elif row['rank_IPI'] <= 500:
        return 'Hotspot preventivo prioritario'
    else:
        return 'Seguimiento'
```

La familia comunica el **tipo de problema** de la zona, no solo su posición en el ranking:

| Familia | Perfil | Tipo de intervención sugerida |
|---|---|---|
| Hotspot robusto integral | Alto en IPI Y en criticidad total | Prioridad máxima: ingeniería vial + control |
| Hotspot de severidad/fatalidad | Pocos siniestros pero muy graves (muertos) | Rediseño de geometría, velocidad, separación de flujos |
| Hotspot de carga acumulada | Alto volumen, severidad baja | Gestión de tráfico, señalización, operaciones |
| Hotspot preventivo prioritario | Señales incipientes, no alcanza umbrales altos | Prevención, educación vial, monitoreo |
| Seguimiento | Sin señales críticas relevantes | Monitoreo rutinario |

#### c) `tipo_hotspot` — clasificación temprana por umbrales absolutos

Esta clasificación se hizo antes del IPI (fase exploratoria del NB02) y usa umbrales absolutos:

```python
def clasificar_hotspot(row):
    if row['anios_activos'] >= 4 and row['cantidad_siniestros'] >= 300:
        return 'Hotspot estructural persistente'
    elif row['criticidad_promedio'] >= 2.0 and row['cantidad_siniestros'] >= 150:
        return 'Hotspot severo'
    elif row['cantidad_siniestros'] >= 350 and row['criticidad_promedio'] < 1.7:
        return 'Hotspot de alto volumen'
    elif row['anios_activos'] >= 3 and row['criticidad_promedio'] >= 1.7:
        return 'Hotspot persistente con severidad media-alta'
    else:
        return 'Hotspot exploratorio'
```

**Relación con la familia analítica:** `tipo_hotspot` es una clasificación más simple y cruda. La `familia_analitica` es la clasificación definitiva del proyecto.

---

### 4.6 — `criticidad_fatalidad_alta`: índice alternativo (1/5/25)

```python
zonas['criticidad_fatalidad_alta'] = (
    zonas['siniestros_solo_danos'] * 1 +
    zonas['siniestros_con_heridos'] * 5 +
    zonas['siniestros_con_muertos'] * 25
)
```

Este índice **no es el principal** del proyecto. Es un instrumento de análisis de sensibilidad: permite ver qué zonas subirían en el ranking si se penalizara más a los siniestros fatales. Con pesos 1/5/25, un muerto equivale a 25 choques materiales (vs 5 con la escala base). Se usa para calcular `variacion_ranking_fatalidad` y detectar las zonas más sensibles a la ponderación.

---

### 4.7 — Zonas sensibles a fatalidad

```python
zonas_sensibles = zonas[
    (zonas['siniestros_con_muertos'] >= 1) &
    (zonas['cantidad_siniestros'] >= 20) &
    (zonas['anios_activos'] >= 2)
]
```

Criterio: zona con al menos un muerto, con masa crítica mínima (20 siniestros) y persistencia (2+ años). Resultado: **2,211 zonas**.

### 4.8 — Concentración espacial y distribución de familias (datos reales)

In [5]:
# Concentración: qué porcentaje del problema captura cada corte del IPI
concentracion = pd.read_csv(REPORTS / 'resumen_concentracion_IPI_2016_2019_final.csv')
print('=== Concentración espacial del IPI — periodo base 2016–2019 ===')
print('(% de siniestros, criticidad y muertes que concentran los cortes Top N)')
display(concentracion)

=== Concentración espacial del IPI — periodo base 2016–2019 ===
(% de siniestros, criticidad y muertes que concentran los cortes Top N)


,top_n_zonas,porcentaje_zonas,siniestros_acumulados,porcentaje_siniestros,criticidad_acumulada,porcentaje_criticidad,siniestros_con_muertos_acumulados,porcentaje_siniestros_con_muertos
0,50,0.2900,4426,1.7000,10138,2.2700,270,6.4300
1,200,1.1700,14304,5.4800,31224,6.9800,760,18.1100
2,500,2.9200,34743,13.3200,69415,15.5300,1622,38.6500
3,1000,5.8400,69604,26.6900,124758,27.9100,2826,67.3300


In [6]:
# Distribución de familias analíticas (post-fix pesos iguales)
familias = pd.read_csv(REPORTS / 'resumen_familia_analitica_2016_2019.csv')
print('=== Distribución de familias analíticas — periodo base 2016–2019 ===')
display(familias)

=== Distribución de familias analíticas — periodo base 2016–2019 ===


,familia_analitica,cantidad_zonas,siniestros_acumulados,criticidad_total_acumulada,muertes_registradas,IPI_promedio
0,Hotspot robusto integral,45,6406,13106,268,92.8794
1,Hotspot de severidad/fatalidad,68,3408,7856,318,92.7550
2,Hotspot preventivo prioritario,346,16850,34862,873,89.9344
3,Hotspot de carga acumulada,155,32449,48741,371,83.9585
4,Seguimiento,16516,201718,342514,2367,50.6286


---

## Sección 5 — NB03: Validación de actualidad

**Objetivo del NB03:** validar si las zonas priorizadas en el periodo base (2016–2019) siguen siendo relevantes con datos más recientes. Explorar las capas hermanas de SIMUR para definir el esquema de integración del NB04.

**Resultado principal:** `outputs/reports/clasificacion_hotspots_persistencia_notebook_03.csv` — 19,255 zonas clasificadas.

---

### 5.1 — Por qué se necesita validación de actualidad

El IPI del NB02 describe el periodo 2016–2019. Presentar esos resultados como diagnóstico del Bogotá actual sería metodológicamente incorrecto (D17, L2). El NB03 responde: **¿cuántas de las zonas críticas del periodo base siguen siendo críticas en los años más recientes disponibles?**

Esta comparación permite distinguir entre:
- **Hotspots persistentes:** zonas estructuralmente problemáticas que no han sido resueltas.
- **Hotspots emergentes:** zonas que no eran críticas antes pero lo son ahora.
- **Hotspots disminuidos:** zonas que eran críticas y dejaron de serlo (posible intervención exitosa, pero también puede ser efecto COVID).

---

### 5.2 — Auditoría de años recientes y decisión ADR-10

Antes de descargar datos recientes, se auditó la integridad de cada año (2020–2025) usando cuatro criterios:
1. Volumen de registros vs periodo base.
2. Cobertura mensual (12/12 meses).
3. Cobertura de localidades (20/20).
4. **Ratio de gravedad:** `CON HERIDOS / SOLO DAÑOS` comparado con el periodo base (~0.49).

In [7]:
# Auditoría de ratios de gravedad por año
ratios = pd.read_csv(REPORTS / 'auditoria_ratios_gravedad_notebook_03.csv')
print('=== Ratios de gravedad por año (CON HERIDOS / SOLO DAÑOS) ===')
print('Referencia periodo base 2016–2019: ~0.49')
print('Años con ratio >2× la referencia fueron excluidos (ADR-10).')
display(ratios)

=== Ratios de gravedad por año (CON HERIDOS / SOLO DAÑOS) ===
Referencia periodo base 2016–2019: ~0.49
Años con ratio >2× la referencia fueron excluidos (ADR-10).


,anio,n_total,n_solo_danos,n_con_heridos,n_con_muertos,pct_con_heridos,pct_con_muertos,ratio_heridos_danos
0,2016,63655,43014,19512,1129,30.6500,1.7700,0.4536
1,2017,64617,44307,19233,1077,29.7600,1.6700,0.4341
2,2018,66531,42770,22764,997,34.2200,1.5000,0.5322
3,2019,65026,41075,22970,981,35.3200,1.5100,0.5592
4,2020,44049,26797,16589,663,37.6600,1.5100,0.6191
5,2021,28854,17386,11006,462,38.1400,1.6000,0.6330
6,2022,25446,12555,12351,540,48.5400,2.1200,0.9838
7,2023,14115,1115,12454,546,88.2300,3.8700,11.1695
8,2024,14018,1044,12391,583,88.3900,4.1600,11.8688
9,2025,12377,801,10994,582,88.8300,4.7000,13.7253


**Interpretación:** los años 2022–2025 muestran ratios de 11–14× por encima del periodo base. Esto no refleja un cambio real en la siniestralidad — es casi imposible que la proporción de heridos se multiplique 11 veces en un año. La explicación más plausible es un cambio de metodología de registro en SIMUR.

**Decisión (ADR-10):** usar **2020–2021** como periodo de validación. Son los únicos años post-2019 que pasan los 4 criterios de integridad.

**Nota:** 2020–2021 son años de pandemia COVID-19, con movilidad reducida. Los conteos son menores al periodo base (44,049 en 2020; 28,854 en 2021 vs ~65,000/año en la base). Esto no invalida el análisis comparativo, pero **las zonas que disminuyeron pueden haberlo hecho por menor tráfico, no por mejoras en la infraestructura**.

---

### 5.3 — IPI del periodo reciente (2020–2021)

Se repitió exactamente el mismo proceso de NB02 sobre los 72,903 siniestros de 2020–2021:

- Misma agregación espacial (0.001°).
- Mismos 5 scores con percentil de rango.
- **Diferencia en `score_persistencia`:** divisor = 2 (años disponibles: 2020 y 2021), no 4.
- Resultado: **13,417 zonas únicas**.

**Importante — los IPIs no son comparables en magnitud:**
> Un IPI de 80 en el periodo base ≠ un IPI de 80 en el periodo reciente. Cada IPI es un percentil interno a su propio universo de zonas. La comparación válida es de **rankings y presencia en cortes** (Top 50, Top 200), no de valores absolutos.

---

### 5.4 — Clasificación de hotspots por persistencia

In [8]:
# Conteos por categoría de persistencia
clasif = pd.read_csv(REPORTS / 'clasificacion_hotspots_persistencia_notebook_03.csv')
resumen_persistencia = clasif['categoria_persistencia'].value_counts().reset_index()
resumen_persistencia.columns = ['categoria', 'zonas']
print('=== Clasificación de hotspots por persistencia ===')
print(f'Universo total: {len(clasif):,} zonas')
print()
display(resumen_persistencia)
print()
print('Solapamiento Top 200 base vs Top 200 reciente:')
n_persistentes = (clasif['categoria_persistencia'] == 'Persistente').sum()
print(f'  {n_persistentes} zonas persistentes de {200} del Top 200 base → {n_persistentes/200*100:.1f}% de solapamiento')

=== Clasificación de hotspots por persistencia ===
Universo total: 19,255 zonas



,categoria,zonas
0,Sin categoría prioritaria,19020
1,Emergente,165
2,Persistente,35
3,Disminuido,35



Solapamiento Top 200 base vs Top 200 reciente:
  35 zonas persistentes de 200 del Top 200 base → 17.5% de solapamiento


**Definición de categorías:**

| Categoría | Criterio | Interpretación |
|---|---|---|
| **Persistente** | Top 200 en AMBOS periodos (base y reciente) | Zona estructuralmente crítica. Prioridad máxima de intervención. |
| **Emergente** | Top 200 reciente + fuera Top 500 base | Nueva zona crítica. Puede reflejar cambios en movilidad post-pandemia. |
| **Disminuido** | Top 50 base + fuera Top 500 reciente | Reducción significativa. ⚠️ No necesariamente indica mejora real. |
| **Histórico** | Top 50 base + sin actividad en reciente | Zona que dejó de registrar siniestros completamente. |
| **Sensible fatalidad** | ≥1 muerto en cualquier periodo | Zonas con al menos un siniestro fatal (puede solapar con otras categorías). |

**Advertencia crítica sobre las zonas disminuidas:** la reducción en el ranking puede deberse al efecto COVID-19 (menos tráfico = menos siniestros), no a intervenciones viales efectivas. Sin datos de exposición (volumen vehicular), no es posible separar ambos efectos.

---

### 5.5 — Capas hermanas SIMUR auditadas en NB03

In [9]:
# Capas SIMUR auditadas
capas = pd.read_csv(REPORTS / 'auditoria_capas_hermanas_simur_notebook_03.csv')
print('=== Auditoría de capas hermanas SIMUR (NB03) ===')
display(capas)

=== Auditoría de capas hermanas SIMUR (NB03) ===


,capa_id,nombre,n_registros,nota,url_query
0,0,MUERTO,15637,"15,637",https://sig.simur.gov.co/arcgis/rest/services/...
1,1,LESIONADO,474962,"474,962",https://sig.simur.gov.co/arcgis/rest/services/...
2,2,ACCIDENTE,900302,"900,302",https://sig.simur.gov.co/arcgis/rest/services/...
3,3,VM_ACC_ACTOR_VIAL,3267230,"3,267,230",https://sig.simur.gov.co/arcgis/rest/services/...
4,4,VM_ACC_CAUSA,1865722,"1,865,722",https://sig.simur.gov.co/arcgis/rest/services/...
5,5,VM_ACC_VEHICULO,2803725,"2,803,725",https://sig.simur.gov.co/arcgis/rest/services/...
6,6,VM_ACC_VIA,961216,"961,216",https://sig.simur.gov.co/arcgis/rest/services/...


**Capas seleccionadas para NB04:**
- Layer 3 — `VM_ACC_ACTOR_VIAL`: campo `CONDICION` → tipo de actor vial (peatón, motorista, ciclista...).
- Layer 4 — `VM_ACC_CAUSA`: campo `NOMBRE` → causa más frecuente.
- Layer 5 — `VM_ACC_VEHICULO`: campo `CLASE` → tipo de vehículo (moto, automóvil, bus...).
- Layer 6 — `VM_ACC_VIA`: pendiente de verificación (devolvió 0 registros en la muestra de NB03).

---

## Sección 6 — Índices de criticidad: comparación y límites

El proyecto usa cinco índices de criticidad con propósitos distintos. Esta sección los compara sistemáticamente.

---

### Índice 1 — `puntaje_gravedad` (escala ordinal 1/3/5)

**Fórmula:** `SOLO DAÑOS → 1 | CON HERIDOS → 3 | CON MUERTOS → 5`

**Para qué sirve:** asigna un peso a cada siniestro individual. Es la unidad de medida base de todos los demás índices.

**Límites:**
- Escala **ordinal**, no de intervalos. No hay garantía de que la distancia entre 1 y 3 sea igual a la de 3 a 5.
- Subestima la gravedad de los siniestros fatales frente a estándares internacionales (ver tabla en 6.6).
- No captura el número de víctimas por siniestro (un siniestro con 5 muertos cuenta igual que uno con 1).

---

### Índice 2 — `criticidad_total` (suma acumulada)

**Fórmula:** `SUM(puntaje_gravedad)` por zona.

**Para qué sirve:** mide la carga histórica total ponderada de una zona. Es el índice más simple para identificar zonas de alto volumen y gravedad acumulada.

**Límites:**
- **Favorece zonas con muchos siniestros leves.** Ejemplo: una zona con 500 choques materiales tiene `criticidad_total = 500`; una zona con 3 siniestros con muertos tiene `criticidad_total = 15`. La primera aparece 33× más crítica aunque la segunda sea más grave.
- No considera persistencia temporal ni la posición relativa entre zonas.

---

### Índice 3 — `criticidad_promedio` (severidad típica)

**Fórmula:** `MEAN(puntaje_gravedad)` por zona.

**Para qué sirve:** identifica zonas donde los siniestros son típicamente graves, independientemente del volumen total.

**Límites:**
- **No considera volumen.** Una zona con 2 siniestros mortales (`criticidad_promedio = 5.0`) aparece igual que una zona con 200 siniestros mortales (`criticidad_promedio = 5.0`).
- Sensible a zonas con muy pocos registros: una zona con un solo siniestro mortal obtiene el máximo posible (5.0) sin representar un patrón real.

---

### Índice 4 — `criticidad_fatalidad_alta` (escala 1/5/25)

**Fórmula:**
```python
criticidad_fatalidad_alta = solo_danos×1 + con_heridos×5 + con_muertos×25
```

**Para qué sirve:** instrumento de análisis de sensibilidad. Identifica zonas que subirían en el ranking si se penalizara más a los siniestros fatales. No es el índice principal del proyecto.

**Límites:**
- Pesos elegidos arbitrariamente para el análisis. No están validados empíricamente.
- El mismo sesgo de volumen que `criticidad_total`: favorece zonas con muchos siniestros, solo que ahora también los mortales.

---

### Índice 5 — `IPI` (compuesto de 5 dimensiones, 0–100)

**Fórmula:**
```python
IPI = mean([score_volumen,
            score_criticidad_total,
            score_severidad_promedio,
            score_persistencia,
            score_fatalidad]) × 100
```

**Para qué sirve:** es el índice principal del proyecto. Integra las 5 dimensiones más relevantes en una sola métrica comparable. Corrige el sesgo de volumen de los índices simples al incluir severidad promedio y fatalidad como dimensiones independientes.

**Ventajas sobre los índices anteriores:**
- Captura a la vez: volumen, carga total, severidad promedio, persistencia temporal y presencia de muertos.
- La normalización por percentil hace comparables zonas con escalas muy distintas.
- El score de persistencia distingue zonas estructurales (activas todos los años) de zonas accidentales (activas un solo año).

**Límites:**
- Pesos **iguales** entre las 5 dimensiones. Sin evidencia empírica que justifique jerarquización.
- **Sensible a la grilla** elegida (0.001°). Con una grilla diferente (H3, red vial), el Top 50 podría cambiar.
- **No incorpora exposición** (flujo vehicular, población, longitud de red vial). No es un índice de riesgo — es un índice de prioridad exploratoria (L1).
- El IPI de un periodo no es comparable en magnitud con el de otro periodo: son percentiles internos a universos distintos.

---

### 6.6 — Comparación con estándares internacionales de ponderación de gravedad

In [10]:
import pandas as pd

tabla_comparacion = pd.DataFrame([
    {
        'Fuente / Escala': 'VíaSegura AI — puntaje_gravedad (MVP)',
        'Solo daños': 1,
        'Con heridos': 3,
        'Con muertos': 5,
        'Muerto / Daño': '5×',
        'Contexto': 'Escala ordinal simple. Elegida por transparencia y facilidad de comunicación.'
    },
    {
        'Fuente / Escala': 'VíaSegura AI — criticidad_fatalidad_alta (sensibilidad)',
        'Solo daños': 1,
        'Con heridos': 5,
        'Con muertos': 25,
        'Muerto / Daño': '25×',
        'Contexto': 'Instrumento de análisis de sensibilidad. No validado empíricamente.'
    },
    {
        'Fuente / Escala': 'NHTSA EPDO (EE.UU.)',
        'Solo daños': 1,
        'Con heridos': '~8',
        'Con muertos': '~24',
        'Muerto / Daño': '~24×',
        'Contexto': 'Equivalent Property Damage Only. Basado en costos económicos estimados de cada tipo de siniestro.'
    },
    {
        'Fuente / Escala': 'FHWA (costos económicos, EE.UU.)',
        'Solo daños': 1,
        'Con heridos': '~5',
        'Con muertos': '~542',
        'Muerto / Daño': '~542×',
        'Contexto': 'Costos sociales totales (médicos, productividad, calidad de vida). Valor estadístico de vida ~11 MUSD.'
    },
    {
        'Fuente / Escala': 'OMS / ETSC (Europa, QALY)',
        'Solo daños': 1,
        'Con heridos': '10–15',
        'Con muertos': '70–150',
        'Muerto / Daño': '70–150×',
        'Contexto': 'Quality-Adjusted Life Years perdidos. Varía por país y metodología de valoración.'
    },
])

print('=== Comparación de escalas de ponderación de gravedad ===')
display(tabla_comparacion)

=== Comparación de escalas de ponderación de gravedad ===


,Fuente / Escala,Solo daños,Con heridos,Con muertos,Muerto / Daño,Contexto
0,VíaSegura AI — puntaje_gravedad (MVP),1,3,5,5×,Escala ordinal simple. Elegida por transparenc...
1,VíaSegura AI — criticidad_fatalidad_alta (sens...,1,5,25,25×,Instrumento de análisis de sensibilidad. No va...
2,NHTSA EPDO (EE.UU.),1,~8,~24,~24×,Equivalent Property Damage Only. Basado en cos...
3,"FHWA (costos económicos, EE.UU.)",1,~5,~542,~542×,"Costos sociales totales (médicos, productivida..."
4,"OMS / ETSC (Europa, QALY)",1,10–15,70–150,70–150×,Quality-Adjusted Life Years perdidos. Varía po...


**Lectura práctica:** con la escala 1/3/5 del proyecto, **un siniestro fatal equivale a 5 choques materiales**. Con la escala EPDO de la NHTSA (el estándar más usado en análisis de seguridad vial en EE.UU.), equivaldría a **24 choques materiales**. Con los costos económicos de la FHWA, equivaldría a **542**.

Esto significa que el IPI actual probablemente **sub-prioriza zonas con alta fatalidad relativa** frente a zonas con alto volumen de siniestros leves. El análisis de sensibilidad pendiente (ADR-11) evaluará cuánto cambia el Top 50 con escalas alternativas.

---

### 6.7 — Qué mide y qué NO mide el IPI

| | Descripción |
|---|---|
| **SÍ mide** | Prioridad exploratoria de intervención basada en concentración histórica de siniestros |
| **SÍ mide** | Persistencia temporal del patrón de siniestralidad |
| **SÍ mide** | Posición relativa de una zona frente al universo completo de zonas |
| **NO mide** | Riesgo real (falta normalización por flujo vehicular, población, longitud de red) |
| **NO mide** | Causalidad (no dice por qué ocurren los siniestros) |
| **NO mide** | Calidad de la infraestructura vial |
| **NO mide** | Comportamiento del conductor o peatón |
| **NO mide** | El estado actual de la zona (los datos son del periodo 2016–2019) |

**Lenguaje correcto vs lenguaje prohibido (ADR-09):**

| ✅ Correcto | ❌ Prohibido |
|---|---|
| "Zona priorizada para intervención" | "Zona más peligrosa de Bogotá" |
| "Concentración de siniestros" | "Zona de alto riesgo" (sin datos de exposición) |
| "Hotspot persistente en periodo base 2016–2019" | "Esta zona es peligrosa hoy" |
| "Prioridad exploratoria de intervención" | "Riesgo real de accidente" |

---

## Sección 7 — Limitaciones globales del proyecto

| ID | Limitación | Severidad | Estado |
|---|---|---|---|
| **L1** | El IPI no mide riesgo real — falta exposición (flujo, población, longitud red vial) | Alta | Aceptada. Documentada en lenguaje correcto del proyecto. |
| **L2** | El periodo base es 2016–2019, no el Bogotá actual | Alta | Aceptada. Toda comunicación debe enmarcar el periodo explícitamente. |
| **L3** | Datos 2022–2025 con ratio de gravedad anómalo (cambio metodológico SIMUR) | Alta | Documentada. Esos años excluidos del análisis (ADR-10). |
| **L4** | Grilla de 0.001° arbitraria — no respeta la red vial | Media | Aceptada para MVP. Comparación con H3/DBSCAN pendiente. |
| **L5** | Pesos 1/3/5 no validados empíricamente — subestiman la fatalidad | Media | Documentada. Análisis de sensibilidad pendiente (ADR-11). |
| **L6** | Filtro bounding box no confirmado con valores mínimos/máximos exactos del raw | Baja | Aceptada. Resultado observado: 0 filas eliminadas. |
| **L7** | Sin integración de actores viales ni vehículos en el IPI base | Media | En proceso. NB04 corrige esto para el periodo reciente. |
| **L8** | Sin hash MD5 del raw descargado inicialmente | Baja | Parcialmente resuelta. `scripts/generar_snapshot_metadata.py` creado. |
| **L9** | Periodo de validación 2020–2021 afectado por COVID-19 | Media | Aceptada. Las zonas "disminuidas" no implican mejora real sin datos de exposición. |
| **L10** | Bug ADR-06 en NB02 corregido post-NB03 — NB03 ejecutado con IPI anterior | Baja | Delta máx. ~0.03 pts. No afecta NB04. Reejecutar NB03 antes de NB05. |

---

## Sección 8 — Hoja de ruta completa hacia NB04–NB06

### Estado de los notebooks

| Notebook | Nombre | Estado | Outputs principales |
|---|---|---|---|
| NB01 | Exploración y validación de datos SIMUR | ✅ Completado | `accidentes_bogota_2016_2019_limpio.csv` (260,831 registros) |
| NB02 | Índice de Prioridad de Intervención (IPI) | ✅ Completado + fix | `top50_IPI_final_2016_2019.csv`, `mapa_top50_IPI_final_2016_2019.html` |
| NB03 | Validación de actualidad + exploración SIMUR | ✅ Completado | `clasificacion_hotspots_persistencia_notebook_03.csv`, `top200_IPI_reciente_notebook_03.csv` |
| NB03.5 | Síntesis metodológica (este notebook) | ✅ Creado | Solo lectura — sin outputs nuevos |
| **NB04** | **Enriquecimiento: actores, vehículos, causas** | ⏳ **PRÓXIMO** | `hotspots_enriquecidos_nb04.csv`, `mapa_hotspots_enriquecidos_nb04.html` |
| NB05 | Normalización por exposición (población, red vial) | Pendiente | Tasas de siniestralidad por 10,000 hab. y por km de red vial |
| NB06 | Dashboard Streamlit + síntesis final | Pendiente | `app/` Streamlit, informe PDF, README final |

---

### NB04 — Insumos disponibles

| Archivo | Descripción | Filas |
|---|---|---|
| `outputs/reports/clasificacion_hotspots_persistencia_notebook_03.csv` | 19,255 zonas clasificadas | 19,255 |
| `outputs/reports/top200_IPI_reciente_notebook_03.csv` | Top 200 zonas del periodo reciente | 200 |
| `data/processed/accidentes_bogota_reciente_limpio.csv` | Siniestros 2020–2021 con FORMULARIO | 72,903 |
| `outputs/reports/esquema_integracion_nb04_notebook_03.csv` | Contrato de integración: capas, campos, cardinalidad | 4 capas |

### NB04 — Secciones propuestas

```
Sec 1 — Setup y validación de insumos NB03
Sec 2 — Índice FORMULARIO → celda de grilla
         {formulario: (round(lat, 3), round(lon, 3))}
Sec 3 — Descarga VM_ACC_ACTOR_VIAL (layer 3)
         Campo objetivo: CONDICION → actor predominante
Sec 4 — Descarga VM_ACC_VEHICULO (layer 5)
         Campo objetivo: CLASE → vehículo predominante
Sec 5 — Descarga VM_ACC_CAUSA (layer 4)
         Campo objetivo: NOMBRE → causa predominante
Sec 6 — Verificación VM_ACC_VIA (layer 6)
         Investigar por qué devolvió 0 en la muestra
Sec 7 — Join y agregación por zona
         Para cada zona Top 200: FORMULARIOs → join capas → moda
Sec 8 — Enriquecimiento del DataFrame de clasificación
         + actor_predominante, vehiculo_predominante, causa_predominante
         + mapa interactivo con popup completo
Sec 9 — Manifiesto y resumen ejecutivo
```

### NB05 — Fuentes externas necesarias

- **Población:** DANE — proyecciones por UPZ o localidad.
- **Red vial:** OpenStreetMap (osmnx) o capa oficial (IDECA/IGAC).
- **Tasas:** siniestros por 10,000 habitantes y por km de red vial.

### NB06 — Producto final

- Dashboard Streamlit en `app/` con mapa interactivo y filtros.
- Informe técnico PDF.
- README final con instrucciones de reproducción completa.
- Snapshot con hash MD5 de todas las fuentes usadas.